In [3]:
from datetime import datetime
from dotenv import load_dotenv
load_dotenv()


import re
from memvid_sdk import use, FindResult

from shared import embedder

# A .mv2 file only allows one exclusive writer handle at a time, so every user's
# ConversationMemory must share a single open handle instead of each opening its
# own (which would raise LockedError: MV007 as soon as a second user connects).
# Per-user isolation is still enforced below via uri/scope, not by the handle.
_shared_mem = use("langchain", "knowledge.mv2", enable_lex=True, enable_vec=True, mode="auto")


class ConversationMemory:
    def __init__(self, user_id: str):
        self.mem = _shared_mem
        self.user_id = user_id
        # Trailing slash matters: find(scope=...) does plain string-prefix
        # matching, so "user/alice" (no slash) would also match "user/alice2/...".
        self.scope = f"user/{user_id}/"
        self.session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        self._message_seq = 0


    def add_message(self, role: str, content: str):
        self._message_seq += 1
        self.mem.put_many(
            [{
                "title": f"Conversation - {self.session_id}",
                "label": "conversation",
                "labels": ["conversation"],
                "text": f"[{role.upper()}]: {content}",
                "uri": f"{self.scope}{self.session_id}/{self._message_seq}",
                "metadata": {
                    "session_id": self.session_id,
                    "user_id": self.user_id,
                    "role": role,
                    "timestamp": datetime.now().isoformat(),
                },
            }],
            embedder=embedder,
        )


    def get_relevant_context(self, query: str, k: int = 5):
        # Scoped call: only this user's own prior messages (isolation enforced
        # by the search engine's URI-prefix filter, not just app-side checks).
        own_results: FindResult = self.mem.find(query, k=k, embedder=embedder, scope=self.scope)
        # Unscoped call: shared knowledge base, explicitly excluding anything
        # labeled "conversation" so no other user's chat history can leak in here.
        knowledge_results: FindResult = self.mem.find(query, k=k, embedder=embedder)

        context_parts: list[str] = []
        for hit in own_results["hits"]:
            context_parts.append(f"[Previous conversation] {hit['text']}")
        for hit in knowledge_results["hits"]:
            if "conversation" not in hit.get("labels", []):
                context_parts.append(f"[Knowledge] {hit['text']}")

        return "\n\n".join(context_parts)


    def get_recent_messages(self, limit: int = 10):
        # timeline() has no server-side scope/uri filter, so over-fetch and
        # filter client-side to this user's own uri prefix before truncating.
        timeline = self.mem.timeline(limit=max(limit * 20, 200), reverse=True)
        messages = []

        for entry in timeline:
            if not entry.get("uri", "").startswith(self.scope):
                continue
            match = re.match(r"^\[(USER|ASSISTANT)\]:\s*(.*)$", entry.get("preview", ""), re.DOTALL)
            if not match:
                continue
            role, content = match.groups()
            messages.append({
                "role": role.lower(),
                "content": content,
                "timestamp": entry.get("timestamp"),
            })
            if len(messages) >= limit:
                break

        return list(reversed(messages))


In [ ]:
_shared_mem.find("""ИНСТРУКЦИЯ ПОСТАВЩИКА
ПО РАБОТЕ С МАШИНОЧИТАЕМЫМИ
ДОВЕРЕННОСТЯМИ""", embedder=embedder, mode="sem")

{'question': 'ИНСТРУКЦИЯ ПОСТАВЩИКА\nПО РАБОТЕ С МАШИНОЧИТАЕМЫМИ\nДОВЕРЕННОСТЯМИ',
 'answer': 'исунок 7), которые описаны в разделе 1.1. Рисунок 7 – «Операции с МЧД» кнопки действия, администратор title: 1.2 Профиль компании «Операции с МЧД» (part 3) labels: manual section: "1.2" section_title: "Профиль компании «Операции с МЧД»" source: "Инструкция_по_работе_с_машиночитаемыми_доверенностями.pdf" [1] нность, которая ни разу не использовалась. Рисунок 3 – «Операции с МЧД» кнопки действия, пользователь Рисунок 4 – Модальное окно «Полномочия» Ознакомится с перечнем полномочий можно в ЕСНСИ https://esnsi.gosuslugi.ru/classifiers Для работы на Портале поставщиков применяются следующие полномочия для МЧД: - Контракт: дей [2] ости в формате *. xml; \uf02d кнопка «Обновить» (Рисунок (3)) - при нажатии обновляет статус выбранной доверенности, если доверенность была отозвана, то статус поменяется; \uf02d номер доверенности ( Рисунок 3(4)) – при нажатии открывает модальное окно «Полномочия» (Рису

In [2]:
from langchain.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from pathlib import Path
import os
class Chatbot:

    def __init__(self, user_id: str):
        self.llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0.7,
            api_key=os.environ.get("OPENAI_API_KEY"),
        )
        self.memory = ConversationMemory(user_id)
        self.system_prompt = open(Path("system_prompt.md")).read()



    def chat(self, message: str):
        context = self.memory.get_relevant_context(message)
        self.memory.add_message("user", message)
        messages = [
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=f"""Context from knowledge base and previous conversations:
{context}

---

User message: {message}

Please respond helpfully based on the context above.""")
        ]


        response =self.llm.invoke(messages)
        assistant_message = response.content

        self.memory.add_message("assistant", assistant_message)
        return assistant_message
        



In [3]:
from dotenv import load_dotenv

load_dotenv()
bot = Chatbot(user_id="demo_user")


In [ ]:
user_id = input("User ID: ").strip() or "anonymous"
bot = Chatbot(user_id=user_id)

while True:
    print("hello:")
    user_input = input("\nYou: ").strip()
    if not user_input:
        continue

    elif user_input == '/history':
        messages = bot.memory.get_recent_messages(10)
        print("\nRecent conversation:")
        for msg in messages:
            print(f" [{msg['role']}] {msg['content'][:100]}...")
    
    else:
        response = bot.chat(user_input)
        print(f"\nBot: {response}")



hello:

Bot: Hello! How can I assist you today with your account work or any specific inquiries?
hello:

Bot: The owner of the UnitedHealth Group account is Priya Shah. If you need any more information about this account or anything else, feel free to ask!
hello:

Recent conversation:
 [user] Hello
title: Conversation - 20260911_170808
uri: user/anonymous/20260911_170808/1
tags: hello user
l...
 [assistant] Hello! How can I assist you today with your account work or any specific inquiries?
title: Conversat...
 [user] Who is the owner of the UnitedHealth Group?
title: Conversation - 20260911_170808
uri: user/anonymou...
 [assistant] The owner of the UnitedHealth Group account is Priya Shah. If you need any more information about th...
hello:
hello:
hello:
hello:
hello:
hello:
hello:
hello:


In [7]:
# This query is can't run properly because this RAG doesn't have a text2SQL layer. In SQL this query will be easy.
# query = "What is all amount employee count of the accounts?"
# Another example, when questions not in the context of knowledge base he doesn't answer on it.
# First of all need search at the query context and the apply decision on it.
# query = "How are you?"
query = "Who is the owner of the UnitedHealth Group?"
results = bot.memory.mem.find(query, embedder=embedder, mode="auto")

In [8]:
results

{'query': 'Who is the owner of the UnitedHealth Group?',
 'hits': [{'frame_id': 26,
   'uri': 'mv2://frames/26',
   'title': 'UnitedHealth Group',
   'rank': 1,
   'score': 0.5861252546310425,
   'matches': 1,
   'snippet': 'Account of UnitedHealth Group company that uses rental-car programs. - Annual Recurring Revenue in dollars: 4400961.96 - Annual Recurring Revenue in dollars band: 1m-5m - Owner: Priya Shah - Signed contract at: 2025-01-11 - Employee count band: 5001+ - Size class: strategic - Region: NA - Trajectory: expanding - Industry: Healthcare - Contracted rental-vehicle capacity: 6410 - Average number of activated rental cars: 1.233 - Health score: 93 - Renewal date: 2027-01-31 title: ...',
   'text': 'Account of UnitedHealth Group company that uses rental-car programs. - Annual Recurring Revenue in dollars: 4400961.96 - Annual Recurring Revenue in dollars band: 1m-5m - Owner: Priya Shah - Signed contract at: 2025-01-11 - Employee count band: 5001+ - Size class: strategic - 

In [2]:
from dotenv import load_dotenv

load_dotenv()


import pandas as pd
from memvid_sdk import use

from shared import embedder

path = "learnings.mv2"
learning_mem = use("basic", path, enable_lex=True, enable_vec=True)


In [11]:
# This query is can't run properly because this RAG doesn't have a text2SQL layer. In SQL this query will be easy.
# query = "What is all amount employee count of the accounts?"
# Another example, when questions not in the context of knowledge base he doesn't answer on it.
# First of all need search at the query context and the apply decision on it.
# query = "How are you?"
query = "Who is the owner of the UnitedHealth Group?"
results = learning_mem.find(query, embedder=embedder, mode="auto")


In [ ]:
results

{'query': 'Who is the owner of the UnitedHealth Group?',
 'hits': [{'frame_id': 1,
   'uri': 'mv2://frames/1',
   'title': 'BigQuery acme_corp warehouse map with observed value domains',
   'rank': 1,
   'score': 0.4744507670402527,
   'matches': 1,
   'snippet': 'BigQuery dataset acme_corp is the sales system of record (48 accounts; query via bigquery_execute_query). accounts: health_score, trajectory steady/expanding/at_risk/new, arr_usd, renewal_date, owner_csm. account_contacts: role champion/economic_buyer/technical/executive, is_decision_maker. account_renewals: term_months, auto_renew, status. account_product_usage: monthly mau/revenue/contracted_seats per sku. products: category program/addon/services/support, tier, list_price_...',
   'text': 'BigQuery dataset acme_corp is the sales system of record (48 accounts; query via bigquery_execute_query). accounts: health_score, trajectory steady/expanding/at_risk/new, arr_usd, renewal_date, owner_csm. account_contacts: role champion/

In [ ]:
learning_mem.tools

/home/bevzd/workspace/GraphRag/.venv/lib/python3.13/site-packages/IPython/utils/dir2.py:67: RuntimeWarning: basic kind exposes no tools
  canary = getattr(obj, "_ipython_canary_method_should_not_exist_", None)


In [1]:
from boring_semantic_layer.agents.tools import BSLTools
from pathlib import Path

def _build_tool_definitions():
    pass


def _get_system_prompt():
    Path("semantic_layer") / "system.md"


class SemanticLayer(BSLTools):
    def __init__(
        self,
        model_path: Path,
        profile: str | None = None,
        profile_file: Path | str | None = None,
    ):
        super().__init__(model_path, profile, profile_file, "plotext")

    tools = property(lambda self: _build_tool_definitions())

    system_prompt = property(lambda self: _get_system_prompt())

    # def __init__


# tools = BSLTools(
# model_path=Path("db_models.yaml"),
# profile="prod_db",
# profile_file=Path("profiles.yaml"),

# )
